# Model routing with the OpenAI Decisions API as the classifier

Strands' `ModelRouter` picks which model serves each invocation. The built-in
`ClassifierStrategy` uses a chat model to choose; here we plug in a *decision
model* — the OpenAI Decisions API (`gpt-6-luna`) — as the classifier instead. It
answers one bounded choice question ("which candidate fits this request?") with a
confidence and no generated text, so the routing decision is fast and cheap.

This mirrors `model_routing_with_jev.py`: only the strategy changes, from Jev to
OpenAI Decisions. The router is passed as the agent's model (the Strands docs:
*pass the router as the agent's model; do not attach it through the plugin
list*).

The point of this notebook is to **see the routing decision** — which model the
Decisions API picks for each request, and with what confidence — not just the
agent's final answer.

**Setup:** AWS creds for the candidate Bedrock models; `OPENAI_API_KEY` and
`pip install "openai>=3.26.0"`.

In [ ]:
import os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("AWS_REGION", "us-east-1")
from dotenv import load_dotenv
load_dotenv()

## The routing strategy

One bounded `choice` over the router's candidates, via the async OpenAI client.
It records each decision in `self.decisions` — `(chosen_model, confidence)` — so
we can see *why* each request went where it did. It routes the opening attempt
only and declines (`None`) after a failed serving attempt, so the router
surfaces the error rather than cycling.

In [ ]:
from typing import Any
from openai import AsyncOpenAI
from strands.models import RoutingCandidate, RoutingContext, RoutingStrategy

class OpenAIDecisionsRoutingStrategy(RoutingStrategy):
    """Route each invocation with the OpenAI Decisions API: one bounded choice
    over the candidates. Records every decision so it can be inspected."""

    def __init__(self) -> None:
        self._oai = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.decisions = []   # (chosen_model, confidence) per routed request

    async def select(self, context: RoutingContext, **kwargs: Any) -> RoutingCandidate | None:
        if context.attempts:            # only choose the opening candidate
            return None
        query = ""
        for m in reversed(context.messages):
            if m.get("role") == "user":
                query = "".join(b.get("text", "") for b in m.get("content", []) if "text" in b)
                break
        choices = [{"value": c.name, "description": c.description or c.name} for c in context.candidates]
        resp = await self._oai.decisions.create(
            model="gpt-6-luna", input=query,
            questions=[{"type": "choice", "name": "model",
                        "instructions": "Which model should handle this request?", "choices": choices}],
        )
        answer = next(a for a in resp.answers if getattr(a, "name", None) == "model")
        self.decisions.append((answer.choice, answer.confidence))
        for c in context.candidates:
            if c.name == answer.choice:
                return c
        return None

## Route an agent, and show each decision

Two candidates: a small model (`routine`) and a stronger one (`advanced`). For
each question we print the model the Decisions API chose and its confidence,
then the answer the chosen model produced.

In [ ]:
from strands import Agent
from strands.models import BedrockModel, ModelRouter, RoutingCandidate

def build_agent(strategy):
    routine = RoutingCandidate(
        BedrockModel(model_id="us.amazon.nova-lite-v1:0", region_name=os.environ["AWS_REGION"]),
        name="routine",
        description="Short factual questions and routine requests.",
    )
    advanced = RoutingCandidate(
        BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=os.environ["AWS_REGION"]),
        name="advanced",
        description="Multi-step reasoning, systems design, troubleshooting.",
    )
    router = ModelRouter(models=[routine, advanced], strategy=strategy)
    return Agent(model=router, callback_handler=None)

strategy = OpenAIDecisionsRoutingStrategy()
agent = build_agent(strategy)

QUESTIONS = [
    "What time zone is Tokyo in?",
    "Design a rollback-safe migration from regional to global idempotency keys.",
]

for q in QUESTIONS:
    before = len(strategy.decisions)
    r = await agent.invoke_async(q)
    chosen, confidence = strategy.decisions[before]   # the decision made for this request
    print(f"Q: {q}")
    print(f"   routed to: {chosen}  (confidence {confidence})")
    print(f"   answer:    {str(r)[:90]}\n")

## The decisions, on their own

Every routing choice the Decisions API made, in order.

In [ ]:
for (q, (chosen, conf)) in zip(QUESTIONS, strategy.decisions):
    print(f"{chosen:9s} (confidence {conf})  <-  {q}")